# 04 - Encode Symile CXRs with RAD-DINO

The Symile arrays are already ImageNet-normalized. This notebook reverses that normalization, reconstructs RGB images, applies the RAD-DINO processor, and stores one frozen image embedding per row. CheXpert labels are not encoder inputs.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
from PIL import Image
import torch
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
SYMILE_ROOT = Path(config["symile_root"])
FEATURE_ROOT = ARTIFACT_ROOT / "features"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
BATCH_SIZE = int(os.environ.get("RADDINO_BATCH_SIZE", "16"))
SPLITS = os.environ.get("BRIDGE_FEATURE_SPLITS", ",".join(config["splits"])).split(",")
print({"device": str(DEVICE), "batch_size": BATCH_SIZE, "splits": SPLITS})


In [ ]:
processor = AutoImageProcessor.from_pretrained(config["cxr_model_id"], cache_dir=config["hf_cache"])
encoder = AutoModel.from_pretrained(
    config["cxr_model_id"], cache_dir=config["hf_cache"], torch_dtype=DTYPE,
).to(DEVICE).eval().requires_grad_(False)
print("Loaded:", config["cxr_model_id"])


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype="float32")[:, None, None]
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype="float32")[:, None, None]

def to_pil(image):
    array = np.asarray(image, dtype="float32") * IMAGENET_STD + IMAGENET_MEAN
    array = (np.clip(array, 0, 1).transpose(1, 2, 0) * 255).round().astype("uint8")
    return Image.fromarray(array, mode="RGB")

@torch.inference_mode()
def encode_batch(images):
    pixels = processor(images=[to_pil(image) for image in images], return_tensors="pt")["pixel_values"]
    output = encoder(pixel_values=pixels.to(DEVICE, dtype=DTYPE))
    pooled = output.pooler_output if getattr(output, "pooler_output", None) is not None else output.last_hidden_state[:, 0]
    return pooled.float().cpu().numpy()


In [ ]:
def encode_split(split):
    split_dir = FEATURE_ROOT / split
    split_dir.mkdir(parents=True, exist_ok=True)
    cxr = np.load(SYMILE_ROOT / "data_npy" / split / f"cxr_{split}.npy", mmap_mode="r")
    hadm = np.load(SYMILE_ROOT / "data_npy" / split / f"hadm_id_{split}.npy", mmap_mode="r")
    unique_hadm, first_index, inverse = np.unique(np.asarray(hadm), return_index=True, return_inverse=True)
    probe = encode_batch(cxr[first_index[:1]])
    unique = np.lib.format.open_memmap(split_dir / "cxr_raddino_unique.npy", mode="w+",
                                        dtype="float32", shape=(len(unique_hadm), probe.shape[1]))
    unique[:1] = probe
    for start in tqdm(range(1, len(unique_hadm), BATCH_SIZE), desc=f"RAD-DINO {split}"):
        stop = min(start + BATCH_SIZE, len(unique_hadm))
        unique[start:stop] = encode_batch(cxr[first_index[start:stop]])
        unique.flush()
    output = np.lib.format.open_memmap(split_dir / "cxr_raddino.npy", mode="w+",
                                        dtype="float32", shape=(len(cxr), probe.shape[1]))
    for start in range(0, len(cxr), 4096):
        stop = min(start + 4096, len(cxr))
        output[start:stop] = unique[inverse[start:stop]]
    output.flush()
    metadata = {"split": split, "rows": len(cxr), "unique_admissions": len(unique_hadm),
                "embedding_dim": probe.shape[1], "model_id": config["cxr_model_id"],
                "pooling": "pooler_output_or_cls", "complete": True}
    (split_dir / "cxr_raddino_metadata.json").write_text(json.dumps(metadata, indent=2))
    print(split, output.shape)

for split in SPLITS:
    encode_split(split)


In [ ]:
for split in SPLITS:
    array = np.load(FEATURE_ROOT / split / "cxr_raddino.npy", mmap_mode="r")
    assert all(np.isfinite(array[start:start + 4096]).all() for start in range(0, len(array), 4096))
print("All CXR feature files are finite.")
